# Diffusion-ES: walking through the failure cases

A clean slate for looking at scenarios one at a time and deciding what to change. Two initial pools only:

* `baseline`: production, 64 i.i.d. diffusion samples.
* `fps_x4`: draw 4×64 diffusion samples and keep 64 by farthest-point sampling on xy at 4 waypoints (same model, spread over its modes).

Everything else is the best config so far: e450 world model (`expected_independent`) + `margin_tiebreak` reward, P=64, 8 elites, 4 ES iterations.

**Per scenario:** run both closed loops (Section 3), look at what happened (3.1), then at the decisions before the failure: was a safe plan in the initial pool, did ES keep it, and what did the WM predict (Section 4). Write down what you see in the notes cell at the end of Section 4.

In [ ]:
import os
# Torch WM + JAX planner share one GPU; must be set before JAX is imported.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import tensorflow as tf
tf.config.set_visible_devices([], "GPU")

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import es_debug as esd
print("JAX compilation cache:", esd.enable_compilation_cache())  # restarted kernels skip most of the jit warm-up

## 1. Configuration

In [ ]:
SUBSET_DIR = "/zfsauton/scratch/sbellad/Diffusion_ES_sim/es_failure_subset_n1500_seed0"
TFRECORD_DIR = f"{SUBSET_DIR}/tfrecords/training"
LANE_GRAPH_DIR = "/zfsauton/scratch/mineuih/waymax_rs/lane_graphs"

# ---- models: best config so far (e450 WM + margin_tiebreak) -----------------
DIFFUSION_CKPT = "/zfsauton/scratch/sbellad/train_diffusion/diffusion_baseline_baseline_20260810_144559/epoch_0500"
WM_CKPT = "/zfsauton/scratch/ishanj/world_model/ckpts_multifuture_gaussian/multiFuture_GaussianWorldModel_20261005_092950/epoch_0450.pt"
WM_BACKEND = "multifuture"
WM_PLANNING_MODE = "expected_independent"
REWARD = esd.margin_tiebreak_reward()

# ---- ES / sim (= slurm_es_fail1500_wm_mf.sh) --------------------------------
ES = dict(population_size=64, elite_size=8, num_iterations=4, resample_timesteps=5)
START_TIMESTEP = 10
REPLAN_INTERVAL = 10
SIM_AGENTS = "log"     # use "idm" for the reactive cases
SEED = 0

### Candidates

Clean scenarios (not offroad/colliding at t=10, goal > 20 m away) that **fail in all four ES runs** (latest.pt binary / tiebreak, e450 tiebreak idm / log), i.e. not seed noise. Labels are the e450 tiebreak log outcome.

In [ ]:
CANDIDATES = {
    # collision in every ES run (47 total); 00009#388 / 00017#173 still reach the goal -> clean dodge tests
    "collision": [("00009", 388), ("00017", 173), ("00001", 243), ("00013", 133), ("00023", 131)],
    # offroad in every ES run although the logged ego never leaves the road (21 total)
    "offroad":   [("00011", 168), ("00091", 145), ("00092", 26), ("00106", 375), ("00124", 425)],
    # goal missed in every ES run (163 total)
    "goal_miss": [("00002", 0), ("00002", 70), ("00003", 331), ("00006", 57)],
    # collide only with IDM agents -> run with sim_agents="idm"
    "reactive":  [("00101", 237), ("00152", 363)],
}

def tfrecord_path(shard: str) -> str:
    return f"{TFRECORD_DIR}/training_tfexample.tfrecord-{shard}-of-01000"

## 2. Load models

In [ ]:
policy, preprocess_cfg = esd.load_diffusion_policy(DIFFUSION_CKPT, seed=SEED)
wm = esd.load_world_model(WM_CKPT, backend=WM_BACKEND, planning_mode=WM_PLANNING_MODE)

baseline_planner = esd.DebugESPlanner(
    policy, preprocess_cfg,
    other_agent_forecast="wm", world_model_forecaster=wm,
    reward=REWARD, acquisition=esd.topk, initial_pool=esd.standard_pool, label="baseline",
    **ES,
)
variants = {
    "baseline": baseline_planner,
    "fps_x4": baseline_planner.variant(label="fps_x4", initial_pool=esd.oversample_fps(4)),
}

## 3. One scenario: closed loop

Pick a candidate. Both variants run from t=10 with the same seed; the baseline run also defines the decisions inspected in Section 4.

In [ ]:
CATEGORY, (SHARD, SCENARIO_IDX) = "collision", CANDIDATES["collision"][1]   # <- pick one
sim_agents = "idm" if CATEGORY == "reactive" else SIM_AGENTS

scenario = esd.load_scenario(tfrecord_path(SHARD), SCENARIO_IDX, lane_graph_dir=LANE_GRAPH_DIR)
loop_kw = dict(start_timestep=START_TIMESTEP, replan_interval_steps=REPLAN_INTERVAL,
               sim_agents=sim_agents, seed=SEED, verbose=False)
runs = {name: esd.run_closed_loop(planner, scenario, **loop_kw) for name, planner in variants.items()}
for r in runs.values():
    print(r.report(), "| first failure t:", esd.first_failure_t(r))
base = runs["baseline"]

In [ ]:
esd.plot_timeline(base, compare=[r for k, r in runs.items() if k != "baseline"]);

### 3.1 What happened

* **Replans:** one panel per replan. Agents at the replan time, executed ego path over the next interval (green), chosen plan (red), last ES elites (blue), goal (star). Red titles = a failure in that interval.
* **Animation:** scrub through the rollout with the plan and elites at each replan.
* **Speed:** executed speed (thick) and the planned speed at each replan (thin). A jump at a replan means the planner switched plans.

In [ ]:
V = "baseline"   # <- variant
esd.plot_closed_loop_replans(runs[V]);

In [ ]:
esd.animate_closed_loop(runs[V], view_m=40, show_plans=True, show_elites=True, show_ids=True)

In [ ]:
esd.plot_speed_profile(list(runs.values()));

## 4. Decisions before the failure

The plan executing at the first failure and the three replans before it (from the baseline run). For each decision and variant, `iter 0` = initial pool, last iter = after ES:

* `n_safe`: plans with no WM mode hitting (collision term ≥ 0.99) and on-road.
* `n_safe` = 0 at iter 0, > 0 later: ES found a safe plan by itself.
* `n_safe` > 0 at iter 0, 0 later: ES lost it (elites chosen elsewhere).
* `n_safe` > 0 at the end but the run still fails: the WM / scorer judged the plan safe but the rollout disagrees, so look at the forecast (4.2).
* `n_safe` = 0 throughout: no sampled plan avoids the predicted conflict; either the pool never covers the escape or none exists any more (check an earlier decision).

In [ ]:
t_fail = esd.first_failure_t(base)
fail_anchor = base.anchor_for(t_fail if t_fail is not None else base.replan_steps[-1])
ANCHORS = [s for s in base.replan_steps if fail_anchor - 3 * REPLAN_INTERVAL <= s <= fail_anchor]
print("t_fail:", t_fail, "anchors:", ANCHORS)

rows, traces = [], {}
for anchor in ANCHORS:
    for name, planner in variants.items():
        tr = base.traces[anchor] if name == "baseline" else esd.replan_at(planner, base, anchor, seed=SEED)
        traces[(anchor, name)] = tr
        for it in (0, -1):
            rows.append({"anchor": anchor, "variant": name, **esd.pool_stats(tr, it)})
pool_df = pd.DataFrame(rows).set_index(["anchor", "variant", "iter"])
pool_df[["n_safe", "frac_collision_free", "frac_onroad", "best_score",
         "endpoint_std_across_m", "endpoint_std_along_m", "final_speed_std_mps", "heading_change_std_deg"]].round(2)

### 4.1 ES at one decision

Pool coloured by score per iteration, elites bold, chosen plan red. Orange = agents the chosen plan hits with P ≥ 5% (solid = their WM top-1 future); dots on it and on the red plan mark the same instants every 1 s.

In [ ]:
A = ANCHORS[-1]   # <- decision to inspect
for name in variants:
    esd.plot_es_iterations(traces[(A, name)], base.snapshots[A], ego_idx=scenario.ego_idx, view_m=35)
    plt.show()

In [ ]:
# Chosen plan's per-agent risk, and the top of the final pool with every reward term.
V = "baseline"   # <- variant
display(esd.risky_agents(traces[(A, V)]))
display(esd.candidates_table(traces[(A, V)], iteration=-1, top=15))

### 4.2 What the world model predicted

All WM modes per agent (alpha ∝ mode probability), the logged future (black dashed) and the chosen plan. Compare with what the agents actually did in the rollout (3.1): if the risky agent's real path is not among the modes, the failure is a forecast miss, not a sampling or ranking problem.

In [ ]:
esd.plot_wm_forecast(traces[(A, V)], base.snapshots[A], ego_idx=scenario.ego_idx, view_m=40);

In [ ]:
esd.animate_wm_forecast(traces[(A, V)], base.snapshots[A], ego_idx=scenario.ego_idx, view_m=40)

### Notes for this scenario

* Failure type / time:
* Was a safe plan sampled (which decision, which variant)?
* Did ES keep it?
* Did the WM forecast the conflict?
* What would have avoided it:

## 5. Sweep over the candidates

Closed loop for both variants on every candidate, plus the pool stats at the decisions before the baseline failure. A few minutes per scenario and variant; trim `CANDIDATES` first if needed.

In [ ]:
def run_case(category, shard, idx, seed=SEED):
    sc = esd.load_scenario(tfrecord_path(shard), idx, lane_graph_dir=LANE_GRAPH_DIR)
    kw = dict(loop_kw, sim_agents="idm" if category == "reactive" else SIM_AGENTS, seed=seed)
    b = esd.run_closed_loop(baseline_planner, sc, **kw)
    t_fail = esd.first_failure_t(b)
    fa = b.anchor_for(t_fail if t_fail is not None else b.replan_steps[-1])
    anchors = [s for s in b.replan_steps if fa - 2 * REPLAN_INTERVAL <= s <= fa]
    pool_rows, loop_rows = [], []
    for name, planner in variants.items():
        for a in anchors:
            tr = b.traces[a] if name == "baseline" else esd.replan_at(planner, b, a, seed=seed)
            p0, pl = esd.pool_stats(tr, 0), esd.pool_stats(tr, -1)
            pool_rows.append({"category": category, "scenario": f"{shard}#{idx}", "variant": name, "anchor": a,
                              "n_safe_init": p0["n_safe"], "n_safe_final": pl["n_safe"],
                              "spread_init_m": p0["endpoint_pairwise_m"], "across_init_m": p0["endpoint_std_across_m"]})
        r = b if name == "baseline" else esd.run_closed_loop(planner, sc, **kw)
        loop_rows.append({"category": category, "scenario": f"{shard}#{idx}", "variant": name,
                          "success": r.success, "overlap": r.overlap, "offroad": r.offroad,
                          "goal_reached": r.goal_reached, "min_goal_dist_m": round(r.min_goal_distance_m, 1),
                          "first_fail_t": esd.first_failure_t(r)})
    return pool_rows, loop_rows

POOL, LOOP = [], []
for category, cases in CANDIDATES.items():
    for shard, idx in cases:
        print(f"--- {category} {shard}#{idx}", flush=True)
        try:
            p, l = run_case(category, shard, idx)
        except Exception as e:  # keep the sweep going
            print("   failed:", repr(e))
            continue
        POOL += p; LOOP += l
        print("   " + "  ".join(f"{r['variant']}={'OK' if r['success'] else 'fail'}" for r in l), flush=True)

In [ ]:
# Diagnosis: did the baseline initial pool contain a safe plan at the decisions before the failure?
pool = pd.DataFrame(POOL)
diag = (pool[pool.variant == "baseline"]
        .groupby(["category", "scenario"])
        .agg(min_safe_init=("n_safe_init", "min"), min_safe_final=("n_safe_final", "min"),
             spread_init_m=("spread_init_m", "mean")))
diag["verdict"] = np.where(diag.min_safe_init == 0, "no safe plan sampled",
                  np.where(diag.min_safe_final == 0, "safe plan lost by ES", "safe plan available (scoring/WM)"))
display(diag)
display(diag.verdict.value_counts())
display(pool.pivot_table(index=["category", "scenario", "anchor"], columns="variant",
                         values=["n_safe_init", "n_safe_final"]).round(1))

In [ ]:
loop = pd.DataFrame(LOOP)
display(loop.pivot_table(index=["category", "scenario"], columns="variant", values="success", aggfunc="first"))
display(loop.groupby("variant")[["success", "overlap", "offroad", "goal_reached"]].mean().round(2))